# Paper 40 backbone on YOLO26n: MobileNetV4 (conv-small) backbone

| Part | Source |
|---|---|
| **Backbone (layers 0–8)** | **Paper 40:** YOLO26n layers 0–8 replaced by **MobileNetV4-Conv-Small** (ImageNet pretrained) + 1×1 channel adapters |
| Backbone (layers 9–10) | YOLO26n base SPPF and C2PSA, unchanged |
| Neck FPN + PAN (layers 11–22) | YOLO26n base, unchanged (SimAM of Paper 40 is not used here) |
| Head (Detect, layer 23) | YOLO26n base, unchanged (one-to-one, NMS-free) |

The baseline ("Standard YOLO26n Backbone") is **not trained here**; its values are already filled in the final table.

**Kaggle settings:** Accelerator = **GPU T4 x1**, Internet = **On** (needed for the ImageNet MobileNetV4 weights), keep the notebook **Private**

## 1. Setup

In [1]:
# ultralytics version is pinned: use the same version on Kaggle, Raspberry Pi 5 and Jetson
!pip install -q ultralytics==8.4.137 roboflow timm==1.0.15

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 5.6 MB/s eta 0:00:00


In [2]:
import os, sys, gc, shutil, random
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch, torchvision
import ultralytics

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| torchvision", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (turn on the GPU accelerator)")

WORK = Path("/kaggle/working")
os.chdir(WORK)
sys.path.insert(0, str(WORK))  # so mnv4_backbone.py can be imported

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
ultralytics 8.4.137 | torch 2.11.0+cu128 | torchvision 0.26.0+cu128
GPU: Tesla T4


## 2. Dataset (download from Roboflow)

The Roboflow API key is written in the cell below. Keep this notebook **Private** on Kaggle.

In [3]:
import os, yaml
from roboflow import Roboflow

os.chdir("/kaggle/working")

# Roboflow API key (keep this notebook Private on Kaggle)
API_KEY = "CxHgc1Nn3FK5vYNsBtJW"

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("md-mahibullah").project("digitalaccess-2-copy")
dataset = project.version(4).download("yolo26")  # Roboflow YOLO26 export (same label format as YOLOv8)

# Rename and fix paths
new = "/kaggle/working/digitalaccess"
if os.path.exists(dataset.location) and not os.path.exists(new):
    os.rename(dataset.location, new)

# Update YAML
yaml_path = f"{new}/data.yaml"
with open(yaml_path) as f:
    data = yaml.safe_load(f)
data["path"], data["train"], data["val"] = new, "train/images", "valid/images"
if os.path.exists(f"{new}/test/images"):
    data["test"] = "test/images"
else:
    data.pop("test", None)
with open(yaml_path, "w") as f:
    yaml.dump(data, f)

# the following cells use these variables
DATA_ROOT = Path(new)
DATA_YAML = yaml_path
names = data["names"]
assert data.get("test"), "No test split: the evaluation and FPS steps need a test split"

for s in ["train", "valid", "test"]:
    print(f"{s:6s}: {len(list((DATA_ROOT / s / 'images').glob('*')))} images")
print("nc =", len(names))
print("✅ Dataset prepared!")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to digitalaccess-2(copy)-4 in yolo26:: 100%|██████████| 17613/17613 [00:01<00:00, 11606.42it/s]

train : 7704 images
valid : 550 images
test  : 550 images
nc = 27
✅ Dataset prepared!


## 3. Paper 40 backbone: MobileNetV4

**What changes** (Paper 40, Section III-B). The convolutional feature extractor of YOLO26n is replaced by
**MobileNetV4-Conv-Small** (timm `mobilenetv4_conv_small.e2400_r224_in1k`, ImageNet pretrained). It uses inverted
bottlenecks and depthwise convolutions only (no attention), so it exports to NCNN with standard layers.

**How it is connected (the neck stays YOLO26n base).**

| Layer | Base YOLO26n | Paper 40 backbone | Output to |
|---|---|---|---|
| 0 | Conv | **MobileNetV4 body** (returns the stride-8, 16, 32 features) | layers 4, 6, 8 |
| 1, 2, 3, 5, 7 | Conv / C3k2 | pass-through (no computation) | — |
| 4 | C3k2 (P3, 128 ch) | **1×1 adapter**: MNv4 stride-8 feature → 128 ch | neck (concat at layer 15) |
| 6 | C3k2 (P4, 128 ch) | **1×1 adapter**: MNv4 stride-16 feature → 128 ch | neck (concat at layer 12) |
| 8 | C3k2 (P5, 256 ch) | **1×1 adapter**: MNv4 stride-32 feature → 256 ch | SPPF (layer 9) |
| 9, 10 | SPPF, C2PSA | unchanged | neck |

The adapters give the neck exactly the channel counts it had before, so **no layer index changes and every neck and
head weight of `yolo26n.pt` still fits**.

**Pretrained weights.** `yolo26n.pt` is loaded first (SPPF, C2PSA, neck and head keep their trained weights), then layers
0–8 are swapped: MobileNetV4 starts from its ImageNet weights, and only the three 1×1 adapters start from scratch.

**Extra package.** `timm` is needed in Kaggle and on the Jetson (to load the `.pt`); it is **not** needed on the
Raspberry Pi 5 (NCNN model).

**Assumptions** (not stated in Paper 40): the MobileNetV4 variant (conv-small) and its ImageNet weights, keeping SPPF and
C2PSA, and the 1×1 adapters to the YOLO26n channel counts.

In [4]:
%%writefile mnv4_backbone.py
"""Paper 40 backbone for YOLO26n: layers 0-8 -> MobileNetV4-Conv-Small (timm, ImageNet) + 1x1 adapters; SPPF, C2PSA, neck, head = base."""
import timm
import torch.nn as nn

from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.modules.conv import Conv
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import RANK

MNV4_NAME = "mobilenetv4_conv_small.e2400_r224_in1k"
TAPS = {4: 0, 6: 1, 8: 2}  # YOLO26n layer -> index of the MobileNetV4 feature (stride 8, 16, 32)
PASS_LAYERS = (1, 2, 3, 5, 7)


class MNV4Body(nn.Module):
    """MobileNetV4 feature extractor returning the stride-8, 16 and 32 feature maps."""

    def __init__(self, pretrained=True):
        super().__init__()
        self.body = timm.create_model(MNV4_NAME, pretrained=pretrained, features_only=True, out_indices=(2, 3, 4))
        assert self.body.feature_info.reduction() == [8, 16, 32], self.body.feature_info.reduction()
        self.channels = self.body.feature_info.channels()

    def forward(self, x):
        return tuple(self.body(x))


class PassThrough(nn.Module):
    """Keeps a removed YOLO26n layer index free (returns its input unchanged)."""

    def forward(self, x):
        return x


class Tap(nn.Module):
    """Takes one MobileNetV4 feature from layer 0 and maps it to the YOLO26n channel count with a 1x1 Conv (BN + SiLU)."""

    def __init__(self, k, c1, c2):
        super().__init__()
        self.k = k
        self.adapt = Conv(c1, c2, 1, 1)

    def forward(self, feats):
        return self.adapt(feats[self.k])


def _set(model, i, new, f, name):
    old = model.model[i]
    dev = next(old.parameters()).device if any(True for _ in old.parameters()) else next(model.parameters()).device
    new = new.to(dev)
    new.i, new.f, new.type = i, f, name
    new.np = sum(p.numel() for p in new.parameters())
    model.model[i] = new


def add_mnv4(model, pretrained=True):
    """Replace YOLO26n layers 0-8 with MobileNetV4 + adapters (neck channel counts unchanged). Returns the model."""
    out_ch = {i: model.model[i].cv2.conv.out_channels for i in TAPS}  # 128, 128, 256 for yolo26n
    body = MNV4Body(pretrained)
    _set(model, 0, body, -1, "MNV4Body")
    for i in PASS_LAYERS:
        _set(model, i, PassThrough(), -1, "PassThrough")
    for i, k in TAPS.items():
        _set(model, i, Tap(k, body.channels[k], out_ch[i]), 0, "MNV4Tap")  # f = 0: read the layer-0 output
    model.save = sorted(set(model.save) | {0})  # keep the layer-0 output for the taps
    return model


class MNV4Trainer(DetectionTrainer):
    """Loads yolo26n.pt first (SPPF, C2PSA, neck, head), then swaps layers 0-8 for MobileNetV4 (ImageNet) + adapters."""

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = DetectionModel(cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=False)
        if weights:
            model.load(weights)
        add_mnv4(model)
        if RANK in {-1, 0}:
            print(f"Paper 40 backbone: {MNV4_NAME} (ImageNet) at layers 0-8, feature channels {model.model[0].channels}")
        if verbose and RANK in {-1, 0}:
            model.info()
        return model


Writing mnv4_backbone.py


In [5]:
import importlib, mnv4_backbone
importlib.reload(mnv4_backbone)
from mnv4_backbone import add_mnv4, MNV4Trainer
from ultralytics.nn.tasks import DetectionModel, yaml_model_load
from ultralytics.utils.torch_utils import get_flops

def fused_params_gflops(model, imgsz=640):
    m = deepcopy(model).float().cpu().eval()
    m.fuse(verbose=False)
    return sum(p.numel() for p in m.parameters()) / 1e6, get_flops(m, imgsz)

# structure check (untrained head, ImageNet MobileNetV4): base vs Paper 40 backbone
base = DetectionModel(yaml_model_load("yolo26n.yaml"), nc=len(names), verbose=False).eval()
mnv4 = add_mnv4(deepcopy(base)).eval()
with torch.no_grad():
    x = torch.zeros(1, 3, 640, 640)
    _ = mnv4(x)  # full forward check
    feats = mnv4.model[0](x)
    print("MobileNetV4 feature shapes:", [tuple(f.shape[1:]) for f in feats])
    print("adapter outputs (layers 4, 6, 8):", [tuple(mnv4.model[i](feats).shape[1:]) for i in (4, 6, 8)], "(should be 128x80x80, 128x40x40, 256x20x20)")
print("layers 0-8:", [mnv4.model[i].type for i in range(9)], "| save:", mnv4.save[:6], "...")
for tag, m in (("base", base), ("Paper 40 backbone", mnv4)):
    pm, gf = fused_params_gflops(m)
    print(f"{tag:18s} Params {pm:.4f} M | GFLOPs {gf:.3f}")

Overriding model.yaml nc=80 with nc=27


model.safetensors: reconstructing file:   0%|          |  0.00B / 15.2MB            

model.safetensors: downloading bytes:           |  0.00B            

Unexpected keys (norm_head.num_batches_tracked, classifier.bias, classifier.weight, conv_head.weight, norm_head.bias, norm_head.running_mean, norm_head.running_var, norm_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


MobileNetV4 feature shapes: [(64, 80, 80), (96, 40, 40), (960, 20, 20)]
adapter outputs (layers 4, 6, 8): [(128, 80, 80), (128, 40, 40), (256, 20, 20)] (should be 128x80x80, 128x40x40, 256x20x20)
layers 0-8: ['MNV4Body', 'PassThrough', 'PassThrough', 'PassThrough', 'MNV4Tap', 'PassThrough', 'MNV4Tap', 'PassThrough', 'MNV4Tap'] | save: [0, 4, 6, 10, 13, 16] ...
base               Params 2.3801 M | GFLOPs 5.338
Paper 40 backbone  Params 2.9598 M | GFLOPs 6.093


## 4. Training

Only the Paper 40 backbone model is trained.

In [6]:
from ultralytics import YOLO

EPOCHS  = 100
IMGSZ   = 640
BATCH   = 16
WORKERS = 2
SEED    = 0
PROJECT = str(WORK / "runs")

COMMON = dict(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED, deterministic=True,
    pretrained="yolo26n.pt", device=0, workers=WORKERS, cache=False, patience=30, cos_lr=False,
    project=PROJECT, exist_ok=True, plots=True,
    # channels_last=False: ultralytics' MuSGD crashes on channels_last weights ("view size is not compatible").
    optimizer="MuSGD", lr0=0.01, momentum=0.9, channels_last=False,
)

# (row label in the table, run name)
VARIANTS = [
    ("MobileNetV4 Backbone (Paper 40)", "bb_mnv4"),
]

def best_path(run):
    return Path(PROJECT) / f"{run}_s{SEED}" / "weights" / "best.pt"

for label, run in VARIANTS:
    print(f"\n========== {label} ({run}) ==========")
    model = YOLO("yolo26n.yaml")
    model.train(trainer=MNV4Trainer, **dict(COMMON, name=f"{run}_s{SEED}"))
    del model; gc.collect(); torch.cuda.empty_cache()


========== MobileNetV4 Backbone (Paper 40) (bb_mnv4) ==========
New https://pypi.org/project/ultralytics/8.4.183 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.137 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/digitalaccess/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_widt

Unexpected keys (norm_head.num_batches_tracked, classifier.bias, classifier.weight, conv_head.weight, norm_head.bias, norm_head.running_mean, norm_head.running_var, norm_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


Paper 40 backbone: mobilenetv4_conv_small.e2400_r224_in1k (ImageNet) at layers 0-8, feature channels [64, 96, 960]
YOLO26n summary: 411 layers, 3,092,122 parameters, 3,092,122 gradients, 6.6 GFLOPs
AMP: running Automatic Mixed Precision (AMP) checks...
AMP: downloading yolo26n.pt for AMP checks (one-time, not used for training)...
AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 455.7±179.5 MB/s, size: 27.0 KB)
train: Scanning /kaggle/working/digitalaccess/train/labels... 7704 images, 3 backgrounds, 3 corrupt: 100% ━━━━━━━━━━━━ 7704/7704 1.3Kit/s 5.8s
train: /kaggle/working/digitalaccess/train/images/frame_0046_jpg.rf.66e151f41305eda795c86209eb124a70.jpg: ignoring corrupt image/label: labels mix segment and detection rows
train: /kaggle/working/digitalaccess/train/images/frame_0046_jpg.rf.6eb9d6d5d174d96f22219c80e2e972cb.jpg: ignoring corrupt image/label: labels mix segment and detection rows
train: /kaggle/working/digitalaccess/train/images/frame_0046_jpg.rf.e6

`get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.


      2/100      3.77G      1.377      5.166     0.0189         14        640: 100% ━━━━━━━━━━━━ 482/482 4.8it/s 1:41
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 7.3it/s 2.5s
                   all        550        557      0.595      0.682      0.697      0.371

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      3/100      3.78G      1.439      2.426    0.01963         12        640: 100% ━━━━━━━━━━━━ 482/482 4.9it/s 1:38
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 7.5it/s 2.4s
                   all        550        557      0.716      0.754      0.831      0.479

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      4/100      3.78G      1.433      1.714    0.01942          7        640: 100% ━━━━━━━━━━━━ 482/482 4.9it/s 1:37
                 Class     Images  Instances      Box(P 

## 5. Test-set evaluation: Precision, Recall, F1-score, mAP@50, Params, GFLOPs

Params and GFLOPs are counted on the fused inference model, the same way as the baseline row.

In [7]:
import mnv4_backbone  # must be imported before loading a MobileNetV4 checkpoint

rows = []
for label, run in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, train it first"); continue
    r = YOLO(str(w)).val(data=DATA_YAML, split="test", imgsz=IMGSZ, batch=16, device=0, plots=False, verbose=False)
    P, R = float(r.box.mp), float(r.box.mr)
    params, gflops = fused_params_gflops(YOLO(str(w)).model, IMGSZ)
    rows.append({"run": run, "Backbone Variant": label,
                 "Params (M)": round(params, 2), "GFLOPs": round(gflops, 2),
                 "Precision": round(P, 3), "Recall": round(R, 3),
                 "F1-score": round(2 * P * R / (P + R + 1e-9), 3),
                 "mAP@50": round(float(r.box.map50), 3)})
METRICS = pd.DataFrame(rows)
METRICS.drop(columns="run")

Ultralytics 8.4.137 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n summary: 301 layers, 2,959,773 parameters, 0 gradients, 6.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 727.2±240.0 MB/s, size: 37.7 KB)
val: Scanning /kaggle/working/digitalaccess/test/labels... 550 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 550/550 949.5it/s 0.6s
val: New cache created: /kaggle/working/digitalaccess/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 9.9it/s 3.6s
                   all        550        552      0.975      0.933      0.954      0.586
Speed: 0.7ms preprocess, 3.2ms inference, 0.0ms loss, 0.2ms postprocess per image


,Backbone Variant,Params (M),GFLOPs,Precision,Recall,F1-score,mAP@50
0,MobileNetV4 Backbone (Paper 40),2.96,6.09,0.975,0.933,0.953,0.954


## 6. Package for Raspberry Pi 5 and Jetson

Creates `paper40_backbone_edge_bench.zip` in the **Output** tab with:

| Item | Used on |
|---|---|
| `bb_mnv4_ncnn_model/` (NCNN model, imgsz 640) | Raspberry Pi 5 |
| `bb_mnv4.pt` | Jetson |
| `bench_images/` (50 real test images, seed 0) | both |
| `mnv4_backbone.py` | Jetson (needed to load the MobileNetV4 `.pt`, with timm) |
| `pi5_bench.py`, `jetson_bench.py` | benchmark scripts |

Both scripts: batch 1, imgsz 640, cycle through the 50 real test images (loaded into memory first),
100 warm-up runs, then 1000 timed runs end-to-end (preprocessing + inference + postprocessing), and print the **median** ms and FPS.
Unzip into its own folder, e.g. `python -m zipfile -e paper40_backbone_edge_bench.zip paper40_backbone_edge_bench`.

In [8]:
EDGE_IMGSZ = 640
N_BENCH_IMAGES = 50
EDGE = WORK / "paper40_backbone_edge_bench"
shutil.rmtree(EDGE, ignore_errors=True)
EDGE.mkdir()
shutil.copy("mnv4_backbone.py", EDGE / "mnv4_backbone.py")

# 50 real test images (fixed seed, so every model and device uses the same images)
test_imgs = sorted(p for p in (DATA_ROOT / "test" / "images").glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
random.Random(SEED).shuffle(test_imgs)
(EDGE / "bench_images").mkdir()
for p in test_imgs[:N_BENCH_IMAGES]:
    shutil.copy(p, EDGE / "bench_images" / p.name)
print(f"bench_images: {len(list((EDGE / 'bench_images').glob('*')))} real test images")

for _, run in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, skipped"); continue
    shutil.copy(w, EDGE / f"{run}.pt")
    p = YOLO(str(w)).export(format="ncnn", imgsz=EDGE_IMGSZ, half=False)
    shutil.copytree(p, EDGE / f"{run}_ncnn_model")  # ultralytics needs the "_ncnn_model" suffix
    print(f"{run:12s} -> {run}.pt, {run}_ncnn_model/")

COMMON_BENCH = """
import glob, statistics, sys, time
import cv2

IMGSZ, WARMUP, RUNS = {imgsz}, 100, 1000
imgs = [cv2.imread(p) for p in sorted(glob.glob("bench_images/*"))]
imgs = [im for im in imgs if im is not None]
if not imgs:
    sys.exit("No images found in bench_images/. Run this script inside the unzipped folder.")
print(f"{{len(imgs)}} real test images, imgsz {{IMGSZ}}, batch 1, {{WARMUP}} warm-up + {{RUNS}} timed runs (median)")


def bench(model, **kw):
    for i in range(WARMUP):
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
    times = []
    for i in range(RUNS):
        t = time.perf_counter()
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
        times.append((time.perf_counter() - t) * 1000)
    return statistics.median(times)
""".format(imgsz=EDGE_IMGSZ)

PI5 = '''# Raspberry Pi 5 benchmark (NCNN). Run inside the unzipped folder:  python pi5_bench.py
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*_ncnn_model"))
if not models:
    sys.exit("No *_ncnn_model folders found.")
for path in models:
    ms = bench(YOLO(path, task="detect"))
    print(f"{path.replace('_ncnn_model', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

JETSON = '''# Jetson benchmark (PyTorch, CUDA FP16). Run inside the unzipped folder:  python jetson_bench.py
import sys
import torch

if not torch.cuda.is_available():
    sys.exit("WARNING: CUDA GPU not available. Install the NVIDIA Jetson PyTorch wheel for your JetPack, then run again.")
print("GPU:", torch.cuda.get_device_name(0))

sys.path.insert(0, ".")
import mnv4_backbone  # needed to load the MobileNetV4 checkpoint (also needs timm)
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*.pt"))
if not models:
    sys.exit("No .pt files found.")
for path in models:
    ms = bench(YOLO(path, task="detect"), device=0, half=True)
    print(f"{path.replace('.pt', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

(EDGE / "pi5_bench.py").write_text(PI5)
(EDGE / "jetson_bench.py").write_text(JETSON)
shutil.make_archive(str(WORK / "paper40_backbone_edge_bench"), "zip", EDGE)
print("\nDownload paper40_backbone_edge_bench.zip from the Output tab.")

bench_images: 50 real test images
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.137 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ This export format does not support end2end models, disabling the end2end branch.
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO26n summary: 325 layers, 3,084,506 parameters, 0 gradients, 6.1 GFLOPs

PyTorch: starting from '/kaggle/working/runs/bb_mnv4_s0/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 31, 8400) (6.3 MB)
requirements: Ultralytics requirement ['ncnn'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 1 package in 221ms
 Downloaded ncnn
Prepared 1 package in 144ms
Installed 1 package in 4ms
 + ncnn==1.0.20261020

requirements: AutoUpdate success ✅ 1.1s
WARNING ⚠️ requirements: Restart ru

pnnxparam = /kaggle/working/runs/bb_mnv4_s0/weights/best_ncnn_model/model.pnnx.param
pnnxbin = /kaggle/working/runs/bb_mnv4_s0/weights/best_ncnn_model/model.pnnx.bin
pnnxpy = /kaggle/working/runs/bb_mnv4_s0/weights/best_ncnn_model/model_pnnx.py
pnnxonnx = /kaggle/working/runs/bb_mnv4_s0/weights/best_ncnn_model/model.pnnx.onnx
ncnnparam = /kaggle/working/runs/bb_mnv4_s0/weights/best_ncnn_model/model.ncnn.param
ncnnbin = /kaggle/working/runs/bb_mnv4_s0/weights/best_ncnn_model/model.ncnn.bin
ncnnpy = /kaggle/working/runs/bb_mnv4_s0/weights/best_ncnn_model/model_ncnn.py
fp16 = 0
optlevel = 2
device = cpu
inputshape = [1,3,640,640]f32
inputshape2 = 
input = 
input2 = 
customop = 
moduleop = 
get inputshape from traced inputs
inputshape = [1,3,640,640]f32
############# pass_level0
inline module = mnv4_backbone.MNV4Body
inline module = mnv4_backbone.PassThrough
inline module = mnv4_backbone.Tap
inline module = timm.layers.conv_bn_act.ConvNormAct
inline module = timm.layers.norm_act.BatchNormA

NCNN: export success ✅ 11.2s, saved as '/kaggle/working/runs/bb_mnv4_s0/weights/best_ncnn_model' (11.5 MB)

Export complete (12.0s)
Results saved to /kaggle/working/runs/bb_mnv4_s0/weights/best_ncnn_model
Predict:         yolo predict task=detect model=/kaggle/working/runs/bb_mnv4_s0/weights/best_ncnn_model imgsz=640 
Validate:        yolo val task=detect model=/kaggle/working/runs/bb_mnv4_s0/weights/best_ncnn_model imgsz=640 data=/kaggle/working/digitalaccess/data.yaml  
Visualize:       https://netron.app
bb_mnv4      -> bb_mnv4.pt, bb_mnv4_ncnn_model/

Download paper40_backbone_edge_bench.zip from the Output tab.


### How to run on the devices

**Raspberry Pi 5** (same Pi and same conditions as the baseline measurement)
1. Activate the Python environment that has `ultralytics==8.4.137` and `ncnn`.
2. `echo performance | sudo tee /sys/devices/system/cpu/cpu*/scaling_governor`
3. Go into the unzipped folder and run `python pi5_bench.py`.
4. Run it 3 times with a 2–3 minute pause; use the middle value. Check `vcgencmd measure_temp` and `vcgencmd get_throttled` (`throttled=0x0` = OK).

**Jetson**
1. NVIDIA Jetson PyTorch + torchvision for your JetPack, then `ultralytics==8.4.137` and `timm==1.0.15` (needed to load the MobileNetV4 `.pt`).
2. `sudo nvpmodel -m 0` (check with `sudo nvpmodel -q`) and `sudo jetson_clocks`.
3. Go into the unzipped folder and run `python jetson_bench.py`. Run 3 times; use the middle value.

## 7. Final backbone table

The baseline row is already filled in. Type the median FPS of the Paper 40 backbone model
(and the baseline Jetson FPS when you have it), keep `None` for a missing value (shown as `--`), then run the cell.
Run section 5 first in the same session (it fills `METRICS`).

In [9]:
# baseline values (identical YOLO26n model)
BASE_ROW = {"Backbone Variant": "Standard YOLO26n Backbone", "Params (M)": 2.38, "GFLOPs": 5.34,
            "Precision": 0.971, "Recall": 0.937, "F1-score": 0.953, "mAP@50": 0.958,
            "FPS (Rasp-5)": 13.38, "FPS (Jetson)": None}

FPS_PI5    = {"bb_mnv4": None}
FPS_JETSON = {"bb_mnv4": None}

table = METRICS.copy()
table["FPS (Rasp-5)"] = table["run"].map(lambda r: FPS_PI5.get(r))
table["FPS (Jetson)"] = table["run"].map(lambda r: FPS_JETSON.get(r))
COLS = ["Backbone Variant", "Params (M)", "GFLOPs", "Precision", "Recall", "F1-score", "mAP@50", "FPS (Rasp-5)", "FPS (Jetson)"]
table = pd.concat([pd.DataFrame([BASE_ROW]), table[COLS]], ignore_index=True)[COLS]
table = table.astype(object).where(table.notna(), "--")
table.style.hide(axis="index")

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.


Backbone Variant,Params (M),GFLOPs,Precision,Recall,F1-score,mAP@50,FPS (Rasp-5),FPS (Jetson)
Standard YOLO26n Backbone,2.380000,5.340000,0.971000,0.937000,0.953000,0.958000,13.380000,--
MobileNetV4 Backbone (Paper 40),2.960000,6.090000,0.975000,0.933000,0.953000,0.954000,--,--


## 8. Zip all results

Run this last. It creates **`paper40_backbone_all_results.zip`** in the **Output** tab with:

| Folder in the zip | Contents |
|---|---|
| `paper40_backbone_edge_bench/` | NCNN + `.pt` model, 50 bench images, `mnv4_backbone.py`, `pi5_bench.py`, `jetson_bench.py` |
| `weights/` | `best.pt`, renamed `bb_mnv4_best.pt` |
| `runs/` | full training folder: `best.pt`, `last.pt`, `results.csv`, training curves, confusion matrix, PR curves |

In [10]:
import zipfile

ALL_ZIP = WORK / "paper40_backbone_all_results.zip"
folders = [WORK / "paper40_backbone_edge_bench", WORK / "runs"]

with zipfile.ZipFile(ALL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders:
        if not folder.exists():
            print(f"{folder.name}: not found, skipped"); continue
        n = 0
        for f in folder.rglob("*"):
            if f.is_file():
                z.write(f, f.relative_to(WORK))
                n += 1
        print(f"{folder.name + '/':30s} {n} files")
    for _, run in VARIANTS:
        w = best_path(run)
        if w.exists():
            z.write(w, f"weights/{run}_best.pt")
            print(f"weights/{run}_best.pt")

print(f"\nDownload {ALL_ZIP.name} ({ALL_ZIP.stat().st_size / 1e6:.1f} MB) from the Output tab.")

paper40_backbone_edge_bench/   59 files
runs/                          26 files
weights/bb_mnv4_best.pt

Download paper40_backbone_all_results.zip (53.1 MB) from the Output tab.
